### Goal
Build a DPD for the **APA_200MHz** PA (OpenDPD), using the LSTM model of the PA (`results/APA_200MHz/best_model.pt`) as a differentiable digital twin.

Target: `PA(DPD(x)) ≈ G·x`, with G from `apa_linear_gain.ipynb` (`results/APA_200MHz/linear_gain.json`).

```
x ──► DPD (trainable) ──u──► PA_LSTM (frozen) ──► ŷ        loss = MSE(ŷ, G·x)
```

Documentation: `docs/codigo/dpd_lstm.md`, `docs/codigo/dpd.md`

#### Etapa 1 — Load the PA digital twin and validate it
1. Same data and normalization (scalers) as the PA LSTM training
2. LSTM frozen (weights don't change, gradient still flows to the input)
3. Check that it reproduces the measured PA, and how it behaves beyond the training range

#### 1.1 Setup

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from pa_model import Dataset, DATA_RAW, RESULTS
from lstm import ResultsStorage
from dpd import DpdLSTM, GridSearchDPD, load_pa_twin, run_pa, run_dpd
from metrics.metrics import NMSE, EVM, ACEPR
from plots import plot_AM_AM, plot_AM_PM, plot_PSD, COLOR_DATASET, COLOR_LSTM, COLOR_ERROR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print("device:", device)

DATASET = "APA_200MHz"
PA_DIR = RESULTS / DATASET          # PA LSTM grid search (grid_search_lstm.ipynb) and linear_gain.json
DPD_DIR = PA_DIR / "dpd"            # DPD grid search results

spec = json.load(open(DATA_RAW / DATASET / "spec.json"))
FS, BW = spec["input_signal_fs"], spec["bw_main_ch"]

#### 1.2 Data (official split, temporal order)

In [ ]:
train, val, test = Dataset.from_split_csv(DATA_RAW / DATASET)
print(DATASET, "->", len(train), len(val), len(test))

#### 1.3 Loading the frozen PA LSTM

In [ ]:
# Weights, window (WP) and the scalers used in the PA training; dropout zeroed, parameters frozen
pa, WP, scaler_x, scaler_y = load_pa_twin(PA_DIR, device)
print("PA LSTM:", json.load(open(PA_DIR / "best_params.json"))["params"])

#### 1.4 Validation: does the model reproduce the measured PA?

In [ ]:
y_pa = run_pa(pa, test.x, WP, scaler_x, scaler_y, device)
x_ref, y_ref = test.x[WP-1:], test.y[WP-1:]          # aligned with the output of run_pa

print(f"PA LSTM on test: NMSE = {NMSE().compute(y_ref, y_pa):.2f} dB, EVM = {EVM().compute(y_ref, y_pa):.3f} %")

#### 1.5 Extrapolation: how does the model behave beyond the training range?

In [ ]:
alpha = 1.3                                      # input 30% larger than the measured one
y_ext = run_pa(pa, alpha * test.x, WP, scaler_x, scaler_y, device)

fig, ax = plt.subplots(figsize=(10, 5))
plot_AM_AM(x_ref, y_ref, db=False, ax=ax, color=COLOR_DATASET, label="measured PA")
plot_AM_AM(x_ref, y_pa, db=False, ax=ax, color=COLOR_LSTM, label="PA LSTM")
plot_AM_AM(alpha * x_ref, y_ext, db=False, ax=ax, color="tab:green", label=f"PA LSTM with x × {alpha}")
ax.axvline(np.abs(train.x).max(), color="k", ls="--", lw=1)
ax.set_title("AM/AM — validation and extrapolation (dashed: max |x| of the training)")
plt.show()

#### Etapa 2 — Linear gain G
Computed in `apa_linear_gain.ipynb` (least squares on `0 < |x| < 0.7`). No back-off: where |G·x| is larger than what the PA delivers, the DPD has to extrapolate.

In [ ]:
linear_gain = json.load(open(PA_DIR / "linear_gain.json"))
G = linear_gain["gi"] + 1j * linear_gain["gq"]
print(f"G = {G:.4f}  ->  |G| = {np.abs(G):.4f} ({linear_gain['abs_db']:.3f} dB), angle = {linear_gain['angle_deg']:.2f} deg")

# Samples whose target is above the largest output measured in the training (the PA LSTM never saw it)
y_max = np.abs(train.y).max()
n_above = np.sum(np.abs(G * train.x) > y_max)
print(f"max|y| (train) = {y_max:.3f}; train samples with |G·x| above it: {n_above} ({100 * n_above / len(train):.3f} %)")

#### Etapa 3 — Grid search of the DPD
- DPD: `DpdLSTM`, residual (`u = x + f(window of x)`), starts as the identity
- Each training sample is a window of `ws + WP − 1` samples of x: the DPD produces the `WP` samples of u that the PA LSTM needs to output ŷ at the last instant
- Metric used to choose the best combination: validation NMSE between ŷ and G·x
- `dr` fixed at 0 (small model, target without noise); `ws = 1` is the DPD without memory (reference)

In [ ]:
param_grid = {
    "window_size": [1, 5, 10],
    "hidden_size": [8, 16, 32],
    "num_layers": [1, 2],
    "learning_rate": [0.001, 0.0005],
    "dropout": [0.0],
    "batch_size": [128],
}

In [ ]:
# Running the grid search (combinations already done in DPD_DIR are skipped)
gsearch = GridSearchDPD(train, val, pa, WP, scaler_x, scaler_y, G, param_grid, device)
best_metric, best_dpd, best_params = gsearch.run(DPD_DIR)
print("best val NMSE:", best_metric, best_params)

#### Etapa 4 — Results

In [ ]:
# All combinations, best first
storage = ResultsStorage(DPD_DIR, key_metric="val_nmse")
summary = storage.load_summary().sort_values("val_nmse")
summary.head(10)

In [ ]:
# Loading the best DPD
_, params, state_dict = storage.load_best(device)
WD = int(params["ws"])
dpd = DpdLSTM(params["hs"], params["nl"], params["dr"]).to(device)
dpd.load_state_dict(state_dict)
print("best DPD:", params)

# Training history of the best combination
history = storage.load_history(params)
fig, (ax_l, ax_u) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
ax_l.plot(history["train_nmse"], label="train"); ax_l.plot(history["val_nmse"], label="val")
ax_l.set_xlabel("epoch"); ax_l.set_ylabel("NMSE (dB)"); ax_l.set_title("NMSE of PA(DPD(x)) vs G·x"); ax_l.legend()
ax_u.plot(history["train_max_u"], label="train"); ax_u.plot(history["val_max_u"], label="val")
ax_u.axhline(np.abs(train.x).max(), color="k", ls="--", lw=1, label="max |x| of the training")
ax_u.set_xlabel("epoch"); ax_u.set_ylabel("max |u|"); ax_u.set_title("Extrapolation of the DPD output"); ax_u.legend()
for ax in (ax_l, ax_u): ax.grid(True, alpha=.3)
plt.show()

In [ ]:
# Test set: without DPD (PA LSTM alone) and with DPD (DPD + PA LSTM), all aligned with x[WD+WP-2:]
u, y_dpd = run_dpd(dpd, pa, test.x, WD, WP, scaler_x, scaler_y, device)
y_nodpd = run_pa(pa, test.x, WP, scaler_x, scaler_y, device)[WD-1:]
x_t = test.x[WD+WP-2:]
u = u[WP-1:]
target = G * x_t

acepr = ACEPR(FS, BW)
results = {
    "Without DPD": {"NMSE (dB)": NMSE().compute(target, y_nodpd), "EVM (%)": EVM().compute(target, y_nodpd),
                    "ACEPR (dB)": acepr.compute(target, y_nodpd)},
    "With DPD":    {"NMSE (dB)": NMSE().compute(target, y_dpd), "EVM (%)": EVM().compute(target, y_dpd),
                    "ACEPR (dB)": acepr.compute(target, y_dpd)},
}
for name, r in results.items():
    print(f"{name:12s}" + "  ".join(f"{k} = {v:8.3f}" for k, v in r.items()))
print(f"max |u| = {np.abs(u).max():.3f}  (max |x| = {np.abs(x_t).max():.3f})")

In [ ]:
# AM/AM and AM/PM: target G·x, PA without DPD and PA with DPD
fig, (ax_am, ax_pm) = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
for y, color, label in ((target, COLOR_ERROR, "Target G·x"), (y_nodpd, COLOR_DATASET, "Without DPD"), (y_dpd, COLOR_LSTM, "With DPD")):
    plot_AM_AM(x_t, y, db=False, ax=ax_am, color=color, label=label, title="AM/AM")
    plot_AM_PM(x_t, y, db=False, remove_offset=False, ax=ax_pm, color=color, label=label, title="AM/PM")
plt.show()

In [ ]:
# PSD: the DPD should bring the spectrum of the output closer to the target (less power outside the main channel)
plot_PSD({"Target G·x": target, "Without DPD": y_nodpd, "With DPD": y_dpd}, FS, BW,
         colors={"Target G·x": COLOR_ERROR, "Without DPD": COLOR_DATASET, "With DPD": COLOR_LSTM}, title="PSD — test set")

In [ ]:
# Characteristic of the DPD itself: |u| x |x| (above the identity line = expansion to compensate the compression)
fig, ax = plt.subplots(figsize=(7, 7))
plot_AM_AM(x_t, u, db=False, ax=ax, color=COLOR_LSTM, label="DPD output u", y_label="|u|")
lim = max(np.abs(x_t).max(), np.abs(u).max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=1, label="identity u = x")
ax.axhline(np.abs(train.x).max(), color="tab:red", ls=":", lw=1, label="max |x| of the training")
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal"); ax.legend(frameon=False)
ax.set_title("DPD characteristic")
plt.show()